# Step 2 — Government Quality Score (MIPS)

**Business question:** Given a list of Massachusetts provider NPIs, what is each provider's CMS MIPS quality score?

**Logic:** Pull QPP/MIPS final scores, enrich with Care Compare and Part B utilization data, and produce a clean per-NPI table. Business logic (three-situation classification) will be designed after exploring this data.

**Structural ceiling:** All CMS data is Medicare fee-for-service. Providers with mostly commercial patients will look thin in this data. This is a known limitation, not something we can engineer around.

## Step 1 — Load Data

**What this does:** Load the datasets needed for the MIPS government quality assessment:
1. **QPP/MIPS Final Scores** — CMS's official quality assessment per provider
2. **NPPES** — provider registry filtered to Massachusetts (our master NPI list)
3. **Facility Affiliations** — NPI-to-hospital CCN crosswalk
4. **Hospital Quality files** — star ratings, HVBP, HCAHPS, infections, complications, readmissions

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import os
import glob

# --- File detection ---
# QPP/MIPS: try known filename patterns
qpp_candidates = glob.glob("2_datasets/ec_score*.csv") + glob.glob("2_datasets/MIPS*2023*.csv") + glob.glob("2_datasets/*qpp*.csv") + glob.glob("2_datasets/*QPP*.csv")
if not qpp_candidates:
    raise FileNotFoundError(
        "QPP/MIPS file not found. Expected 'ec_score_file.csv' or similar in 2_datasets/. "
        "Download from https://data.cms.gov/provider-data/topics/doctors-clinicians (QPP section)"
    )
qpp_file = qpp_candidates[0]
print(f"Using QPP file: {qpp_file}")

In [ ]:
# Load QPP/MIPS final scores
qpp = pd.read_csv(qpp_file, dtype=str)
print(f"QPP/MIPS: {qpp.shape[0]:,} rows, {qpp.shape[1]} columns")
print(f"Columns: {list(qpp.columns)}")

# Sanity check: expect ~800k-1M rows nationally
if qpp.shape[0] < 500_000:
    print(f"⚠️ WARNING: Only {qpp.shape[0]:,} rows. Expected ~800k-1M. Check if this is the right file/year.")
elif qpp.shape[0] > 2_000_000:
    print(f"⚠️ WARNING: {qpp.shape[0]:,} rows is unusually high. May contain multiple years.")
else:
    print(f"✓ Row count looks reasonable for national MIPS data")

In [ ]:
# Load NPPES — chunked, filter to MA on the fly
nppes_candidates = glob.glob("2_datasets/NPPES Data/npidata_pfile_*.csv")
# Exclude the fileheader file
nppes_candidates = [f for f in nppes_candidates if "fileheader" not in f]
if not nppes_candidates:
    raise FileNotFoundError(
        "NPPES file not found. Expected 'npidata_pfile_*.csv' in 2_datasets/NPPES Data/. "
        "Download from https://download.cms.gov/nppes/NPI_Files.html and extract the ZIP."
    )
nppes_file = nppes_candidates[0]
print(f"Using NPPES file: {nppes_file}")
print(f"File size: {os.path.getsize(nppes_file) / 1e9:.1f} GB")

# Key columns to keep (minimizes memory — full file has 330 columns)
nppes_cols = [
    "NPI",
    "Entity Type Code",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Provider Credential Text",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address Postal Code",
    "Healthcare Provider Taxonomy Code_1",
    "Provider Enumeration Date",
    "NPI Deactivation Date",
]

# Chunked load, filter to MA
print("Loading NPPES and filtering to MA (this takes a few minutes)...")
ma_chunks = []
total_rows = 0
for chunk in pd.read_csv(nppes_file, dtype=str, usecols=nppes_cols, chunksize=50_000):
    total_rows += len(chunk)
    ma_chunk = chunk[
        chunk["Provider Business Practice Location Address State Name"]
        .str.upper().str.strip()
        .isin(["MA", "MASSACHUSETTS"])
    ]
    if len(ma_chunk) > 0:
        ma_chunks.append(ma_chunk)

nppes_ma = pd.concat(ma_chunks, ignore_index=True)
print(f"NPPES total rows processed: {total_rows:,}")
print(f"NPPES MA providers: {nppes_ma.shape[0]:,}")

# Sanity check: expect ~80k-120k MA NPIs
if nppes_ma.shape[0] < 30_000:
    print(f"⚠️ WARNING: Only {nppes_ma.shape[0]:,} MA NPIs. Expected ~80k-120k. Check filter column.")
elif nppes_ma.shape[0] > 200_000:
    print(f"⚠️ WARNING: {nppes_ma.shape[0]:,} MA NPIs is unusually high. Filter may not be working.")
else:
    print(f"✓ MA NPI count looks reasonable")

In [ ]:
# Load CMS Facility Affiliations (NPI -> Hospital CCN crosswalk)
facility_affil = pd.read_csv("2_datasets/Facility_Affiliation.csv", dtype=str,
                              usecols=["NPI", "facility_type", "Facility Affiliations Certification Number"])
print(f"Facility Affiliations: {facility_affil.shape[0]:,} rows")
print(f"\nFacility type distribution:")
print(facility_affil["facility_type"].value_counts().to_string())
print(f"\nUnique NPIs: {facility_affil['NPI'].nunique():,}")
print(f"Unique CCNs: {facility_affil['Facility Affiliations Certification Number'].nunique():,}")

In [ ]:
# Load Hospital Quality files and build per-facility summary
# All keyed by Facility ID (6-digit CCN)

hosp_general = pd.read_csv("2_datasets/Hospital_General_Information.csv", dtype=str,
    usecols=["Facility ID", "Facility Name", "State", "Hospital overall rating", "Hospital Type", "Hospital Ownership"])
print(f"Hospital General Info: {hosp_general.shape[0]:,} facilities")

hcahps = pd.read_csv("2_datasets/HCAHPS-Hospital.csv", dtype=str,
    usecols=["Facility ID", "HCAHPS Measure ID", "Patient Survey Star Rating"])
print(f"HCAHPS: {hcahps.shape[0]:,} rows")

infections = pd.read_csv("2_datasets/Healthcare_Associated_Infections-Hospital.csv", dtype=str,
    usecols=["Facility ID", "Compared to National"])
print(f"Infections: {infections.shape[0]:,} rows")

complications = pd.read_csv("2_datasets/Complications_and_Deaths-Hospital.csv", dtype=str,
    usecols=["Facility ID", "Compared to National"])
print(f"Complications: {complications.shape[0]:,} rows")

readmissions = pd.read_csv("2_datasets/Unplanned_Hospital_Visits-Hospital.csv", dtype=str,
    usecols=["Facility ID", "Compared to National"])
print(f"Readmissions: {readmissions.shape[0]:,} rows")

hvbp = pd.read_csv("2_datasets/hvbp_tps.csv", dtype=str,
    usecols=["Facility ID", "Total Performance Score"])
print(f"HVBP TPS: {hvbp.shape[0]:,} rows")

# Build single hospital quality summary: one row per Facility ID
hosp_summary = hosp_general[["Facility ID", "Facility Name", "State", "Hospital overall rating",
                              "Hospital Type", "Hospital Ownership"]].copy()
hosp_summary = hosp_summary.rename(columns={"Hospital overall rating": "hospital_star_rating"})

# HVBP Total Performance Score
hvbp_scores = hvbp[["Facility ID", "Total Performance Score"]].rename(
    columns={"Total Performance Score": "hvbp_total_score"})
hosp_summary = hosp_summary.merge(hvbp_scores, on="Facility ID", how="left")

# HCAHPS: overall hospital rating star
hcahps_overall = hcahps[hcahps["HCAHPS Measure ID"] == "H_STAR_RATING"][
    ["Facility ID", "Patient Survey Star Rating"]].rename(
    columns={"Patient Survey Star Rating": "hcahps_star_rating"})
hosp_summary = hosp_summary.merge(hcahps_overall, on="Facility ID", how="left")

# Infections: count of "Worse" measures (exact string from data)
inf_worse = infections[infections["Compared to National"] == "Worse than the National Benchmark"]
inf_count = inf_worse.groupby("Facility ID").size().reset_index(name="infection_measures_worse")
hosp_summary = hosp_summary.merge(inf_count, on="Facility ID", how="left")
hosp_summary["infection_measures_worse"] = hosp_summary["infection_measures_worse"].fillna(0).astype(int)

# Complications: count of "Worse" measures (two string variants in data)
comp_worse = complications[complications["Compared to National"].isin([
    "Worse Than the National Rate", "Worse Than the National Value"])]
comp_count = comp_worse.groupby("Facility ID").size().reset_index(name="complication_measures_worse")
hosp_summary = hosp_summary.merge(comp_count, on="Facility ID", how="left")
hosp_summary["complication_measures_worse"] = hosp_summary["complication_measures_worse"].fillna(0).astype(int)

# Readmissions: count of "Worse" measures (two string variants in data)
readm_worse = readmissions[readmissions["Compared to National"].isin([
    "Worse Than the National Rate", "Worse than expected"])]
readm_count = readm_worse.groupby("Facility ID").size().reset_index(name="readmission_measures_worse")
hosp_summary = hosp_summary.merge(readm_count, on="Facility ID", how="left")
hosp_summary["readmission_measures_worse"] = hosp_summary["readmission_measures_worse"].fillna(0).astype(int)

print(f"\nHospital summary: {len(hosp_summary):,} facilities, {hosp_summary.shape[1]} columns")
print(f"Star rating distribution:")
print(hosp_summary["hospital_star_rating"].value_counts().sort_index().to_string())

## Step 2 — Exploratory Data Analysis

**What this does:** Before cleaning or joining anything, understand what we're working with. Schema, shape, null rates, distributions, and key charts for each dataset.

In [ ]:
# --- QPP/MIPS EDA ---
print("=" * 60)
print("QPP/MIPS FINAL SCORES")
print("=" * 60)

print(f"\nShape: {qpp.shape}")
print(f"\nColumns: {list(qpp.columns)}")

# Note: most QPP columns have a leading space in the name
# Strip column names for easier access
qpp.columns = qpp.columns.str.strip()
print(f"\nColumns (cleaned): {list(qpp.columns)}")

print(f"\nNull rates:")
print((qpp.isnull().sum() / len(qpp) * 100).round(1).to_string())

print(f"\nData types:")
print(qpp.dtypes.to_string())

print(f"\nUnique NPIs: {qpp['NPI'].nunique():,}")
print(f"Source distribution:")
print(qpp["source"].value_counts().to_string())

In [ ]:
# QPP visualizations

# MIPS final score distribution
qpp_scores = pd.to_numeric(qpp["final_MIPS_score"], errors="coerce")
fig1 = px.histogram(
    qpp_scores.dropna(),
    nbins=50,
    title="QPP: MIPS Final Score Distribution (National)",
    labels={"value": "MIPS Final Score", "count": "Providers"},
)
fig1.show()

# Category score distributions
category_cols = {
    "Quality_category_score": "Quality",
    "PI_category_score": "Promoting Interoperability",
    "IA_category_score": "Improvement Activities",
    "Cost_category_score": "Cost",
}
for col, label in category_cols.items():
    vals = pd.to_numeric(qpp[col], errors="coerce").dropna()
    if len(vals) > 0:
        fig = px.histogram(vals, nbins=30, title=f"QPP: {label} Category Score Distribution")
        fig.show()
    print(f"{label}: {len(vals):,} non-null values, mean={vals.mean():.1f}, median={vals.median():.1f}")

# No low-volume flag column — will infer from Part B
print("\nNo explicit low-volume flag in QPP data. Will infer from Part B billing volume.")

In [ ]:
# --- NPPES (MA) EDA ---
print("=" * 60)
print("NPPES — MASSACHUSETTS PROVIDERS")
print("=" * 60)

print(f"\nShape: {nppes_ma.shape}")
print(f"\nColumns: {list(nppes_ma.columns)}")

print(f"\nNull rates:")
print((nppes_ma.isnull().sum() / len(nppes_ma) * 100).round(1).to_string())

print(f"\nEntity Type Code distribution:")
print(nppes_ma["Entity Type Code"].value_counts().to_string())

print(f"\nUnique NPIs: {nppes_ma['NPI'].nunique():,}")

# Check for deactivated NPIs
deactivated = nppes_ma["NPI Deactivation Date"].notna().sum()
print(f"\nDeactivated NPIs: {deactivated:,} ({deactivated/len(nppes_ma)*100:.1f}%)")

In [ ]:
# NPPES MA visualizations

# Provider type (Entity Type 1=Individual, 2=Organization)
fig = px.bar(
    nppes_ma["Entity Type Code"].value_counts().reset_index(),
    x="Entity Type Code", y="count",
    title="NPPES MA: Entity Type (1=Individual, 2=Organization)",
)
fig.show()

# Top taxonomy codes (proxy for specialty)
fig = px.bar(
    nppes_ma["Healthcare Provider Taxonomy Code_1"].value_counts().head(20).reset_index(),
    x="Healthcare Provider Taxonomy Code_1", y="count",
    title="NPPES MA: Top 20 Taxonomy Codes",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# City distribution
fig = px.bar(
    nppes_ma["Provider Business Practice Location Address City Name"].value_counts().head(20).reset_index(),
    x="Provider Business Practice Location Address City Name", y="count",
    title="NPPES MA: Top 20 Cities",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# ZIP code distribution
zip_counts = nppes_ma["Provider Business Practice Location Address Postal Code"].str[:5].value_counts().head(25)
fig = px.bar(
    zip_counts.reset_index(),
    x="Provider Business Practice Location Address Postal Code", y="count",
    title="NPPES MA: Top 25 ZIP Codes",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# --- Facility Affiliations EDA ---
print("=" * 60)
print("FACILITY AFFILIATIONS (NPI -> CCN CROSSWALK)")
print("=" * 60)

print(f"\nShape: {facility_affil.shape}")
print(f"\nAffiliations per NPI:")
affil_per_npi = facility_affil.groupby("NPI").size()
print(affil_per_npi.describe().to_string())

print(f"\nNPIs with 1 affiliation: {(affil_per_npi == 1).sum():,}")
print(f"NPIs with 2+ affiliations: {(affil_per_npi >= 2).sum():,}")
print(f"NPIs with 5+ affiliations: {(affil_per_npi >= 5).sum():,}")

In [ ]:
# --- Hospital Quality Summary EDA ---
print("=" * 60)
print("HOSPITAL QUALITY SUMMARY")
print("=" * 60)

print(f"\nShape: {hosp_summary.shape}")
print(f"\nNull rates:")
print((hosp_summary.isnull().sum() / len(hosp_summary) * 100).round(1).to_string())

# Star rating distribution
fig = px.bar(
    hosp_summary["hospital_star_rating"].value_counts().sort_index().reset_index(),
    x="hospital_star_rating", y="count",
    title="Hospital Overall Star Rating Distribution",
)
fig.show()

# HVBP TPS distribution
hvbp_vals = pd.to_numeric(hosp_summary["hvbp_total_score"], errors="coerce").dropna()
if len(hvbp_vals) > 0:
    fig = px.histogram(hvbp_vals, nbins=30, title="HVBP Total Performance Score Distribution")
    fig.show()

# Worse-than-national counts
for col in ["infection_measures_worse", "complication_measures_worse", "readmission_measures_worse"]:
    vals = hosp_summary[col]
    print(f"\n{col}:")
    print(vals.describe().to_string())

## Step 3 — Data Preprocessing & Cleaning

**What this does:** Harmonize NPIs, parse QPP scores, deduplicate, build hospital quality crosswalk, and construct the master table via left joins.

In [ ]:
# --- Step 3: Data Preprocessing & Cleaning ---

# 1. NPI harmonization across all datasets
def clean_npi(df, npi_col="NPI"):
    """Strip whitespace, cast to string, validate 10-digit format."""
    df[npi_col] = df[npi_col].astype(str).str.strip()
    valid_mask = df[npi_col].str.match(r"^\d{10}$")
    invalid_count = (~valid_mask).sum()
    if invalid_count > 0:
        print(f"  {invalid_count:,} invalid NPIs removed (not 10 digits)")
        df = df[valid_mask].copy()
    return df

print("NPI harmonization:")

print(f"\nNPPES MA: {len(nppes_ma):,} rows before")
nppes_ma = clean_npi(nppes_ma)
print(f"  -> {len(nppes_ma):,} rows after")

print(f"\nQPP: {len(qpp):,} rows before")
qpp = clean_npi(qpp, npi_col="NPI")
print(f"  -> {len(qpp):,} rows after")

In [ ]:
# 3. QPP cleaning — parse scores and deduplicate

# Parse MIPS final score to numeric
qpp["mips_final_score"] = pd.to_numeric(qpp["final_MIPS_score"], errors="coerce")

# Parse category scores
qpp["quality_score"] = pd.to_numeric(qpp["Quality_category_score"], errors="coerce")
qpp["cost_score"] = pd.to_numeric(qpp["Cost_category_score"], errors="coerce")
qpp["improvement_activities_score"] = pd.to_numeric(qpp["IA_category_score"], errors="coerce")
qpp["promoting_interoperability_score"] = pd.to_numeric(qpp["PI_category_score"], errors="coerce")

# No low-volume flag in QPP data — will infer from Part B
qpp["low_volume_flag"] = None

print(f"MIPS final score stats:")
print(qpp["mips_final_score"].describe().to_string())

# Deduplication — keep highest MIPS score per NPI
# Trade-off: biases upward. A provider under two TINs may have different scores
# reflecting different practice contexts. For MVP we accept this and document it.
dupes = qpp["NPI"].duplicated().sum()
print(f"\nDuplicate NPIs in QPP: {dupes:,}")
if dupes > 0:
    qpp = qpp.sort_values("mips_final_score", ascending=False).drop_duplicates(subset="NPI", keep="first")
    print(f"After dedup (kept highest score): {len(qpp):,} rows")

In [ ]:
# Hospital Quality attribution via Facility Affiliations (NPI -> CCN -> Hospital Quality)

# Clean facility affiliations NPI
facility_affil = clean_npi(facility_affil, npi_col="NPI")
facility_affil = facility_affil.rename(columns={
    "Facility Affiliations Certification Number": "CCN"
})

# Filter to hospital affiliations only
hosp_affil = facility_affil[facility_affil["facility_type"] == "Hospital"][["NPI", "CCN"]].copy()
print(f"Hospital affiliations: {len(hosp_affil):,} (NPI-CCN pairs)")
print(f"Unique NPIs with hospital affiliation: {hosp_affil['NPI'].nunique():,}")
print(f"Unique hospitals (CCNs): {hosp_affil['CCN'].nunique():,}")

# Join facility affiliations to hospital summary on CCN = Facility ID
hosp_for_provider = hosp_affil.merge(
    hosp_summary[["Facility ID", "hospital_star_rating", "hvbp_total_score",
                   "hcahps_star_rating", "infection_measures_worse",
                   "complication_measures_worse", "readmission_measures_worse"]],
    left_on="CCN",
    right_on="Facility ID",
    how="inner"
).drop(columns=["Facility ID"])

matched_ccns = hosp_for_provider["CCN"].nunique()
print(f"CCNs matched to hospital quality data: {matched_ccns:,}")
print(f"NPI-hospital pairs with quality data: {len(hosp_for_provider):,}")

# A provider may be affiliated with multiple hospitals — keep the one with the best star rating
hosp_for_provider["_star_numeric"] = pd.to_numeric(hosp_for_provider["hospital_star_rating"], errors="coerce")
hosp_for_provider = hosp_for_provider.sort_values("_star_numeric", ascending=False).drop_duplicates(
    subset="NPI", keep="first"
).drop(columns=["_star_numeric"])

# Rename for master join
hosp_for_join = hosp_for_provider.rename(columns={
    "CCN": "affiliated_hospital_ccn",
    "hospital_star_rating": "affiliated_hospital_star_rating",
    "hvbp_total_score": "affiliated_hospital_hvbp_score",
    "hcahps_star_rating": "affiliated_hospital_hcahps_rating",
    "infection_measures_worse": "affiliated_hospital_infection_flags",
    "complication_measures_worse": "affiliated_hospital_complication_flags",
    "readmission_measures_worse": "affiliated_hospital_readmission_flags",
})

print(f"\nHospital attribution ready for join: {len(hosp_for_join):,} unique NPIs")

In [ ]:
# Build master table — NPPES MA as left side, join QPP and Hospital Quality

master = nppes_ma[["NPI", "Entity Type Code", "Provider Last Name (Legal Name)",
                    "Provider First Name", "Provider Credential Text",
                    "Provider Business Practice Location Address City Name",
                    "Provider Business Practice Location Address Postal Code",
                    "Healthcare Provider Taxonomy Code_1"]].copy()

print(f"Master table start: {len(master):,} MA NPIs")

# Left join QPP
qpp_join_cols = ["NPI", "mips_final_score", "low_volume_flag",
                 "quality_score", "cost_score", "improvement_activities_score",
                 "promoting_interoperability_score", "source"]
master = master.merge(qpp[qpp_join_cols], on="NPI", how="left")
qpp_matched = master["mips_final_score"].notna().sum()
print(f"QPP matched: {qpp_matched:,} / {len(master):,} ({qpp_matched/len(master)*100:.1f}%)")

# Left join Hospital Quality (via facility affiliation CCN crosswalk)
master = master.merge(hosp_for_join, on="NPI", how="left")
hosp_matched = master["affiliated_hospital_star_rating"].notna().sum()
print(f"Hospital Quality matched: {hosp_matched:,} / {len(master):,} ({hosp_matched/len(master)*100:.1f}%)")

# Validate: row count should match NPPES MA
assert len(master) == len(nppes_ma), (
    f"Master table has {len(master):,} rows but NPPES MA has {len(nppes_ma):,}. "
    "Left join should preserve all NPPES rows."
)
print(f"\n Master table: {len(master):,} rows (matches NPPES MA count)")

# Column summary
print(f"\nColumn list ({len(master.columns)} total):")
print(list(master.columns))

print(f"\nNull rates:")
print((master.isnull().sum() / len(master) * 100).round(1).sort_values(ascending=False).to_string())

## Output Schema

One row per MA NPI. MIPS score (when available) and hospital affiliation quality signals. This is the Step 2 "government quality assessment" dimension. Other data loaded in this notebook (Part B, Part D, Care Compare, Open Payments) feeds into Steps 3, 4, and other pipeline dimensions.

## Step 4 — Core Transformation: Clean Schema + Confidence Tiers

**What this does:** Extract the Step 2 output from the master table: MIPS scores and hospital affiliation quality only. Compute the data source count and confidence tier. Other data (Part B, Part D, Care Compare, Open Payments) remains in the master table for use by other pipeline steps but is not part of the Step 2 output.

**Confidence tier logic:**
- **Tier 1 (government_assessed):** Has MIPS score — CMS directly evaluated this provider
- **Tier 2 (hospital_signal):** No MIPS, but has hospital affiliation quality data (indirect government assessment)
- **Tier 3 (no_government_assessment):** No MIPS score and no hospital affiliation data

In [ ]:
# --- Step 4: Core Transformation ---

# Build Step 2 output: government quality assessment only
output = pd.DataFrame()

# Identity
output["npi"] = master["NPI"]
output["entity_type"] = master["Entity Type Code"]
output["provider_name"] = (
    master["Provider Last Name (Legal Name)"].fillna("") + ", " +
    master["Provider First Name"].fillna("")
).str.strip(", ")
output["provider_state"] = "MA"  # All providers in this notebook are MA-filtered from NPPES
output["provider_zip"] = master["Provider Business Practice Location Address Postal Code"]
output["taxonomy_code"] = master["Healthcare Provider Taxonomy Code_1"]

# Government Quality Assessment (direct)
output["mips_final_score"] = master["mips_final_score"]
output["mips_quality_score"] = master["quality_score"]
output["mips_cost_score"] = master["cost_score"]
output["mips_ia_score"] = master["improvement_activities_score"]
output["mips_pi_score"] = master["promoting_interoperability_score"]

# Hospital Affiliation Quality (indirect government assessment)
output["affiliated_hospital_ccn"] = master["affiliated_hospital_ccn"]
output["affiliated_hospital_star_rating"] = master["affiliated_hospital_star_rating"]
output["affiliated_hospital_hvbp_score"] = pd.to_numeric(master["affiliated_hospital_hvbp_score"], errors="coerce")
output["affiliated_hospital_hcahps_rating"] = master["affiliated_hospital_hcahps_rating"]
output["affiliated_hospital_infection_flags"] = master["affiliated_hospital_infection_flags"].fillna(0).astype(int)
output["affiliated_hospital_complication_flags"] = master["affiliated_hospital_complication_flags"].fillna(0).astype(int)
output["affiliated_hospital_readmission_flags"] = master["affiliated_hospital_readmission_flags"].fillna(0).astype(int)

print(f"Step 2 output: {output.shape[0]:,} rows, {output.shape[1]} columns")
print(f"Columns: {list(output.columns)}")

In [ ]:
# Compute confidence indicators

# Data source count: MIPS (0 or 1) + Hospital Quality (0 or 1) = 0 to 2
has_mips = output["mips_final_score"].notna().astype(int)
has_hospital = output["affiliated_hospital_star_rating"].notna().astype(int)
output["data_source_count"] = has_mips + has_hospital

# Confidence tier (3 tiers for Step 2)
def assign_tier(row):
    if pd.notna(row["mips_final_score"]):
        return 1  # government_assessed
    elif pd.notna(row["affiliated_hospital_star_rating"]):
        return 2  # hospital_signal
    else:
        return 3  # no_government_assessment

output["confidence_tier"] = output.apply(assign_tier, axis=1)

tier_labels = {1: "government_assessed", 2: "hospital_signal", 3: "no_government_assessment"}
output["confidence_tier_label"] = output["confidence_tier"].map(tier_labels)

# Summary
print(f"Final Step 2 output: {output.shape[0]:,} rows, {output.shape[1]} columns")
print(f"\nConfidence tier distribution:")
for tier in [1, 2, 3]:
    count = (output["confidence_tier"] == tier).sum()
    print(f"  Tier {tier} ({tier_labels[tier]}): {count:,} ({count/len(output)*100:.1f}%)")

print(f"\nData source count distribution:")
print(output["data_source_count"].value_counts().sort_index().to_string())

# Save Step 2 output
output.to_csv("2_datasets/step2_output_final.csv", index=False)
output.to_parquet("2_datasets/step2_output_final.parquet", index=False)
print(f"\nStep 2 output saved to 2_datasets/step2_output_final.csv and .parquet")
print(f"Total columns: {output.shape[1]}")

In [ ]:
# --- Step 2 Output Summary ---
print("=" * 60)
print("STEP 2 OUTPUT: GOVERNMENT QUALITY ASSESSMENT")
print("=" * 60)

print(f"\nShape: {output.shape}")

has_mips = output["mips_final_score"].notna().sum()
has_hosp = output["affiliated_hospital_star_rating"].notna().sum()
has_either = output["data_source_count"].gt(0).sum()
has_neither = output["data_source_count"].eq(0).sum()

print(f"\nGovernment quality coverage:")
print(f"  MIPS score (direct):         {has_mips:>8,} ({has_mips/len(output)*100:>5.1f}%)")
print(f"  Hospital quality (indirect):   {has_hosp:>8,} ({has_hosp/len(output)*100:>5.1f}%)")
print(f"  Either:                     {has_either:>8,} ({has_either/len(output)*100:>5.1f}%)")
print(f"  Neither:                   {has_neither:>8,} ({has_neither/len(output)*100:>5.1f}%)")

if has_mips > 0:
    print(f"\nMIPS score distribution (Tier 1 providers):")
    print(output["mips_final_score"].dropna().describe().to_string())

print(f"\nHospital star rating distribution (where affiliated):")
print(output["affiliated_hospital_star_rating"].value_counts().sort_index().to_string())

## Composite Layer: How to Use Step 2 Output

Step 2 answers one question: **has the government independently assessed this provider's clinical quality?**

### Two types of signal

**Direct (MIPS score):** CMS evaluated this provider's clinical quality across four categories. This is the strongest evidentiary signal in the pipeline. An employer can point to a government-issued quality assessment as part of their fiduciary due diligence. Available for ~3% of the MA provider universe, ~11% of MA physicians.

**Indirect (hospital affiliation quality):** CMS assessed the hospital where this provider practices. This is not the provider's own quality score, but a provider who practices at a hospital with high mortality, infection, or readmission rates is operating in a lower-quality environment. Available for ~11% of the MA provider universe, ~44% of MA physicians.

### Confidence Tiers

| Tier | Label | What the composite should do |
|---|---|---|
| **1** | `government_assessed` | Full weight on this dimension. MIPS score passes through as-is (0-100). |
| **2** | `hospital_signal` | No direct quality score. Hospital affiliation provides indirect context. The composite may give partial weight or use it for cross-referencing only. |
| **3** | `no_government_assessment` | No government quality data. Redistribute this dimension's weight to other steps. This is not a negative signal — it means the provider doesn't treat enough Medicare patients to be evaluated. |

### NULL Handling

- **MIPS fields:** NULL for ~97% of providers. This is the structural ceiling of Medicare FFS data. Never penalize a NULL.
- **Hospital fields:** NULL means no hospital affiliation in CMS records. Common for outpatient-only, office-based providers. Neutral signal.
- The composite should redistribute Step 2's weight when both MIPS and hospital data are NULL.

### Cross-Reference Logic (for the composite, not Step 2)

For Tier 1 providers with low MIPS scores (below 50):
- **Step 1 (Safety Gate):** Low MIPS + clean safety gate = likely administrative burden, not bad care.
- **Step 5 (Patient Experience):** Low MIPS + good reviews = reporting issue. Low MIPS + bad reviews = real quality concern.
- **Hospital affiliation:** Low MIPS + high-rated hospital = more likely a reporting issue.

### Structural Ceiling

All data in this step is Medicare fee-for-service. A physician whose patients are commercially insured will appear "dark" (Tier 3) because of who they treat, not how well they treat them. The composite should treat Tier 3 as an information gap, not a quality gap.

### Data for Other Steps

The full master table (`step2_master_table_full.parquet`, 48 columns) contains cleaned and joined data from all 7 sources. Other pipeline steps should consume it:
- **Step 3 (Utilization):** `partb_*` and `partd_*` columns
- **Step 4 (Credentials):** `care_compare_*` columns (specialty, credential, med school, grad year)
- **Transparency layer:** `open_payments_*` columns

## Step 5 — Proof of Correctness

**Smoke tests** to validate the Step 2 output is correct and complete.

In [ ]:
# --- Step 5: Proof of Correctness ---

# Read back the saved output to verify it round-trips correctly
output_check = pd.read_parquet("2_datasets/step2_output_final.parquet")
print(f"Output file read back: {output_check.shape[0]:,} rows, {output_check.shape[1]} columns")

# Test 1: Row count matches NPPES MA
assert output_check.shape[0] == 250_974 or output_check.shape[0] == len(nppes_ma), \
    f"Row count mismatch: {output_check.shape[0]} vs expected {len(nppes_ma)}"
print("✓ Test 1: Row count matches NPPES MA provider universe")

# Test 2: No duplicate NPIs
assert output_check["npi"].nunique() == len(output_check), \
    f"Duplicate NPIs found: {len(output_check) - output_check['npi'].nunique():,}"
print("✓ Test 2: No duplicate NPIs")

# Test 3: MIPS scores are in valid range (0-100) where present
mips_scores = output_check["mips_final_score"].dropna()
assert mips_scores.between(0, 100).all(), "MIPS scores outside 0-100 range"
print(f"✓ Test 3: All {len(mips_scores):,} MIPS scores in 0-100 range")

# Test 4: Confidence tiers are exhaustive and mutually exclusive
assert set(output_check["confidence_tier"].unique()) <= {1, 2, 3}, \
    f"Unexpected tier values: {output_check['confidence_tier'].unique()}"
tier_counts = output_check["confidence_tier"].value_counts().sort_index()
assert tier_counts.sum() == len(output_check), "Tier counts don't sum to total"
print(f"✓ Test 4: Every NPI has exactly one confidence tier")

# Test 5: Tier 1 providers all have MIPS scores
tier1 = output_check[output_check["confidence_tier"] == 1]
assert tier1["mips_final_score"].notna().all(), "Tier 1 providers missing MIPS scores"
print(f"✓ Test 5: All {len(tier1):,} Tier 1 providers have MIPS scores")

# Test 6: Tier 3 providers have no MIPS and no hospital data
tier3 = output_check[output_check["confidence_tier"] == 3]
assert tier3["mips_final_score"].isna().all(), "Tier 3 providers have MIPS scores"
assert tier3["affiliated_hospital_star_rating"].isna().all(), "Tier 3 providers have hospital data"
print(f"✓ Test 6: All {len(tier3):,} Tier 3 providers have no government data")

# Test 7: Hospital star ratings are valid where present
stars = output_check["affiliated_hospital_star_rating"].dropna()
valid_stars = {"1", "2", "3", "4", "5", "Not Available"}
assert set(stars.unique()) <= valid_stars, f"Unexpected star ratings: {set(stars.unique()) - valid_stars}"
print(f"✓ Test 7: Hospital star ratings are valid values")

# Test 8: Spot check — look up a specific high-scoring provider
top_provider = output_check.sort_values("mips_final_score", ascending=False).iloc[0]
print(f"\n✓ Test 8: Spot check — highest MIPS score:")
print(f"  NPI: {top_provider['npi']}, Score: {top_provider['mips_final_score']:.1f}")
print(f"  Name: {top_provider['provider_name']}")

print(f"\n{'='*60}")
print("ALL SMOKE TESTS PASSED")
print(f"{'='*60}")

## Step 6 — README

See `README.md` in the project root for:
- Dataset download links and source URLs
- Reproduction instructions
- File descriptions

## Step 7 — Push to GitHub

This notebook is committed on branch `ds/step-renumber-and-step6-framework`.